In [ ]:
!pip install pandas
import pandas as pd
import sqlite3
from datetime import datetime

* 1. Create a connection to the database using the sqlite3 library.

In [ ]:
conn = sqlite3.connect('../ex00/data/checking-logs.sqlite')

* 2. Create a new table called datamart in the database by joining the tables pageviews and checker using only one query.

    - The table should have the following columns: "uid", "labname", "first_commit_ts", and "first_view_ts".
    - "first_commit_ts" is a new name for the "timestamp" column in the checker table. It shows the first commit from a particular lab and user.
    - "first_view_ts" shows the first time a user visited the pageviews table. It is the timestamp of when a user visited the newsfeed.
    - status = 'ready' should still be a filter.
    - numTrials = 1 should still be a filter.
    - "labnames" should be from the list: laba04, laba04s, laba05, laba06, laba06s, and project1.
    - The table should contain only users (uids with user_*), not admins.
    - "first_commit_ts" and "first_view_ts" should be parsed as datetime64[ns].


In [ ]:
query = """
SELECT 
    c.uid,
    c.labname,
    MIN(c.timestamp) AS first_commit_ts,
    (
        SELECT MIN(datetime)
        FROM pageviews p
        WHERE p.uid = c.uid
            AND p.uid LIKE 'user_%'
    ) AS first_view_ts
FROM checker c
WHERE c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname
"""

datamart = pd.read_sql(query, conn, parse_dates=['first_commit_ts', 'first_view_ts'])
datamart.to_sql('datamart', conn, if_exists='replace', index=False)
datamart.info()

* 3. Using Pandas methods, create two dataframes: test and control.

    - test should contain users with values in "first_view_ts".
    - control should contain users with missing values in "first_view_ts".
    - Replace the missing values in the control table with the average "first_view_ts" value of the test users. We will use this value for future analyses.
    - Save both tables in the database. You will use them in the next exercises.

In [ ]:
# Создадим test и control
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()
# Заменим NULL в control на среднее из test
mean_view_ts = test['first_view_ts'].mean()
control['first_view_ts'] = control['first_view_ts'].fillna(mean_view_ts)
# Сохраним
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)
# Проверяем, что таблицы создались
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("All tables in database now:")
print(tables)

* 4. Close the connection.

In [ ]:
conn.close()

In [ ]:
test.info()